## Phase 2: Generator Training Data

This notebook filters the full dialogue corpus down to the five characters
selected for this project, then builds prompt-response training pairs used
to fine-tune the persona generator. The five characters, finalized in
Phase 1, are: Jack (Fight Club), Bateman (American Psycho), Alvy (Annie
Hall), Ben (The Graduate), and Erin (Erin Brockovich).

This notebook assumes `data/processed/filtered_lines.csv` and
`config/personas.json` already exist, produced by the
data preparation notebook. The output of this notebook,
`data/processed/training_pairs.csv`, is used directly by the LoRA
fine-tuning step, run separately in Google Colab.

In [4]:
from pathlib import Path
import json
import pandas as pd

PROCESSED_DIR = Path("data/processed")
RAW_DIR = Path("data/raw")

filtered = pd.read_csv(PROCESSED_DIR / "filtered_lines.csv")
with open(Path("config/personas.json")) as f:
    selected_characters = json.load(f)

print(f"Loaded {len(filtered):,} filtered lines")
print(filtered.groupby("persona_tag").size())

Loaded 1,918 filtered lines
persona_tag
alvy       459
bateman    335
ben        432
erin       338
jack       354
dtype: int64


### Step 1: Build prompt-response training pairs

A dialogue line on its own is not enough to teach the model how to reply
in character. The model needs to see actual exchanges: what was said to
the character, and how the character responded. This step uses
`movie_conversations.txt`, which records which lines belong to the same
conversation, to pair each character line with the line that came before
it. The result is a table of (prompt, response, persona_tag) rows that
will be used to fine-tune the generator.

In [5]:
import ast

char_ids = {c["character_id"] for c in selected_characters}
tag_lookup = {c["character_id"]: c["persona_tag"] for c in selected_characters}

full_lines = pd.read_csv(PROCESSED_DIR / "lines.csv")
line_text = dict(zip(full_lines["line_id"], full_lines["text"]))
line_char = dict(zip(full_lines["line_id"], full_lines["character_id"]))

conversations = []
with open(RAW_DIR / "movie_conversations.txt", encoding="latin-1") as f:
    for raw in f:
        fields = raw.split(" +++$+++ ")
        if len(fields) < 4:
            continue
        line_ids = ast.literal_eval(fields[3].strip())
        conversations.append(line_ids)

training_pairs = []
for conv in conversations:
    for i in range(len(conv) - 1):
        prompt_id, reply_id = conv[i], conv[i + 1]
        reply_char_id = line_char.get(reply_id)
        if reply_char_id in char_ids:
            prompt_text = line_text.get(prompt_id, "")
            reply_text = line_text.get(reply_id, "")
            if prompt_text and reply_text:
                training_pairs.append({
                    "persona_tag": tag_lookup[reply_char_id],
                    "prompt": prompt_text,
                    "response": reply_text,
                })

pairs_df = pd.DataFrame(training_pairs)
print(f"Built {len(pairs_df):,} training pairs")
print(pairs_df.groupby("persona_tag").size())

pairs_df.to_csv(PROCESSED_DIR / "training_pairs.csv", index=False)
print(f"\nSaved to {PROCESSED_DIR / 'training_pairs.csv'}")

Built 1,434 training pairs
persona_tag
alvy       344
bateman    255
ben        331
erin       247
jack       257
dtype: int64

Saved to data/processed/training_pairs.csv
